# AML Kafka Consumer

Lê em streaming do tópico Kafka `transactions`, aplica o `PipelineModel` treinado offline e grava
previsões em HDFS (Parquet, particionado pela hora de chegada).

Também escreve métricas agregadas e mostra previsões no console para a defesa.

**Pré-requisitos:** `aml_trainer.ipynb` executado e modelo persistido.

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField,
    TimestampType, IntegerType, StringType, DoubleType,
)
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1 pyspark-shell'

KAFKA_BOOTSTRAP = "10.204.131.11:9092"
KAFKA_TOPIC     = "g12in"
SPARK_MASTER = "spark://10.84.128.47:7077"
HDFS_BASE   = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project"
MODEL_PATH  = f"{HDFS_BASE}/model/rf_aml_pipeline"          #Onde o modelo treinado (Random Forest para AML*) está guardado
PRED_PATH   = f"{HDFS_BASE}/stream/predictions_kafka"       #Onde o Spark escreve as previsões geradas em tempo real a partir do Kafka
METRICS_PATH= f"{HDFS_BASE}/stream/metrics_kafka"           #Onde são escritas as métricas de avaliação do modelo em streaming
CHKPT_PRED  = f"{HDFS_BASE}/stream/chkpt_kafka_predictions" #Checkpoint das previsões — permite retomar o stream sem perda de dados se houver falha
CHKPT_METR  = f"{HDFS_BASE}/stream/chkpt_kafka_metrics"     #Checkpoint das métricas — mesma função, mas para o stream de métricas

spark = SparkSession.builder \
    .master(SPARK_MASTER) \
    .appName("AML Kafka Consumer") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"Kafka bootstrap: {KAFKA_BOOTSTRAP}  topic: {KAFKA_TOPIC}")

Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1305)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1291)
	at org.apache.spark.util.Utils$.fetchFile(Utils.scala:432)
	at org.apache.spark.SparkContext.addFile(SparkContext.scala:1875)
	at org.apache.spark.SparkContext.$anonfun$new$17(SparkContext.scala:551)
	at org.apache.spark.SparkContext.$anonfun$new$17$adapted(SparkContext.scala:551)
	at scala.collection.immutable.List.foreach(List.scala:323)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:551)
	at org.apache.spar

In [20]:
model = PipelineModel.load(MODEL_PATH)
print("Modelo carregado.")

Modelo carregado.


In [21]:
msg_schema = StructType([
    StructField("Timestamp", TimestampType(), True),
    StructField("From Bank", IntegerType(), True),
    StructField("From Account", StringType(), True),
    StructField("To Bank", IntegerType(), True),
    StructField("To Account", StringType(), True),
    StructField("Amount Received", DoubleType(), True),
    StructField("Receiving Currency", StringType(), True),
    StructField("Amount Paid", DoubleType(), True),
    StructField("Payment Currency", StringType(), True),
    StructField("Payment Format", StringType(), True),
    StructField("Is Laundering", IntegerType(), True),
])

raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

parsed = raw.select(
    F.col("timestamp").alias("kafka_ts"),
    F.from_json(F.col("value").cast("string"), msg_schema).alias("d"),
).select("kafka_ts", "d.*")

AnalysisException: Failed to find data source: kafka. Please deploy the application as per the deployment section of Structured Streaming + Kafka Integration Guide.

In [ ]:
# Feature engineering — réplica exacta da do trainer
def add_features(d):
    return (
        d.withColumn("log_amt_paid", F.log1p(F.col("Amount Paid")))
         .withColumn("log_amt_recv", F.log1p(F.col("Amount Received")))
         .withColumn("amt_diff", F.abs(F.col("Amount Paid") - F.col("Amount Received")))
         .withColumn("same_bank", (F.col("From Bank") == F.col("To Bank")).cast("int"))
         .withColumn("ccy_mismatch", (F.col("Receiving Currency") != F.col("Payment Currency")).cast("int"))
         .withColumn("hour", F.hour("Timestamp"))
    )

featured = add_features(parsed).na.fill({"hour": 0})

scored = (
    model.transform(featured)
        .withColumn("prob_fraud", vector_to_array("probability")[1])
        .withColumn(
            "status",
            F.when(F.col("prediction") == 1, F.lit("BLOQUEADA - suspeita de fraude"))
             .otherwise(F.lit("OK - fidedigna")),
        )
        .select(
            "kafka_ts", "Timestamp", "From Bank", "To Bank",
            "From Account", "To Account",
            "Amount Paid", "Receiving Currency", "Payment Format",
            F.col("Is Laundering").alias("truth"),
            F.col("prediction").cast("int").alias("prediction"),
            "prob_fraud", "status",
        )
)

In [ ]:
# Sink 1: previsões em parquet (idempotente via checkpoint)
pred_query = scored \
    .withColumn("ingest_hour", F.date_format("kafka_ts", "yyyy-MM-dd-HH")) \
    .writeStream \
    .outputMode("append") \
    .format("parquet") \
    .partitionBy("ingest_hour") \
    .option("path", PRED_PATH) \
    .option("checkpointLocation", CHKPT_PRED) \
    .trigger(processingTime="5 seconds") \
    .start()

print("Sink de previsões iniciado:", PRED_PATH)

In [ ]:
# Sink 2: métricas agregadas em windows de 30s (throughput + taxa de positivos)
metrics = scored \
    .withWatermark("kafka_ts", "1 minute") \
    .groupBy(F.window("kafka_ts", "30 seconds")) \
    .agg(
        F.count("*").alias("n_msgs"),
        F.sum("prediction").alias("n_pred_pos"),
        F.sum("truth").alias("n_truth_pos"),
    )

metr_query = metrics.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", METRICS_PATH) \
    .option("checkpointLocation", CHKPT_METR) \
    .trigger(processingTime="10 seconds") \
    .start()

print("Sink de métricas iniciado:", METRICS_PATH)

In [ ]:
# Sink 3: consola — útil para a defesa
console_query = scored.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .option("numRows", 20) \
    .trigger(processingTime="10 seconds") \
    .start()

spark.streams.awaitAnyTermination()

In [ ]:
# Parar manualmente (executar quando a demo terminar)
for q in spark.streams.active:
    print("A parar:", q.name or q.id)
    q.stop()